In [ ]:
import torch
import os
import pandas as pd
from torch_geometric.data import HeteroData
from datetime import datetime
import numpy as np
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # 启用详细错误报告
os.environ['TORCH_USE_CUDA_DSA'] = '1'   # 启用设备端断言

# 1. 获取所有日期列表
node_dir = "../data_processed/all/nodes"
all_dates = sorted(
    [f.split('_')[1].split('.')[0] for f in os.listdir(node_dir) 
    if f.startswith('news_') and f.endswith('.csv')
])
num_days = len(all_dates)
split_idx = int(num_days * 2 / 3)
train_dates = all_dates[:split_idx]
test_dates = all_dates[split_idx:]

print(f"总天数: {num_days}")
print(f"训练集日期: {train_dates[0]} 至 {train_dates[-1]} ({len(train_dates)}天)")
print(f"测试集日期: {test_dates[0]} 至 {test_dates[-1]} ({len(test_dates)}天)")

# 2. 为每个日期创建独立的异构图
def create_daily_graph(date):
    """为指定日期创建异构图"""
    data = HeteroData()
    
    # 读取三个文件
    node_file = f"../data_processed/all/nodes/news_{date}.csv"
    ss_edge_file = f"../data_processed/all/SS_edge/news_{date}.csv"
    ns_is_edge_file = f"../data_processed/all/NS_IS_edge/news_{date}_with_MR_with_SA.csv"
    
    if not all(os.path.exists(f) for f in [node_file, ss_edge_file, ns_is_edge_file]):
        print(f"跳过不完整日期: {date}")
        return None
    
    # 读取数据
    nodes_df = pd.read_csv(node_file)
    ss_edge_df = pd.read_csv(ss_edge_file)
    ns_is_edge_df = pd.read_csv(ns_is_edge_file)
    
    # 创建本地映射
    stocks = pd.concat([ss_edge_df['stock_1'], ss_edge_df['stock_2']]).unique()
    industries = ns_is_edge_df['industry_code'].unique()
    news_ids = ns_is_edge_df['id'].unique()
    
    stock_map = {code: idx for idx, code in enumerate(stocks)}
    industry_map = {code: i for i, code in enumerate(industries)}
    news_map = {id: i for i, id in enumerate(news_ids)}
    
    # 添加股票节点
    stock_features = []
    stock_labels = []
    for _, row in nodes_df.iterrows():
        if row['ts_code'] in stock_map:
            stock_idx = stock_map[row['ts_code']]
            # 确保特征维度一致
            features = row[2:10].values.astype(np.float32)
            if len(features) < 8:
                features = np.pad(features, (0, 8 - len(features)), 'constant')
            stock_features.append(features)
            stock_labels.append(row['label'])
    
    if stock_features:
        data['stock'].x = torch.tensor(np.array(stock_features), dtype=torch.float)
        data['stock'].y = torch.tensor(stock_labels, dtype=torch.long)
    
    # 添加行业节点（无特征）
    data['industry'].x = torch.ones(len(industry_map), 1, dtype=torch.float)  # 占位特征
    data['industry'].num_nodes = len(industry_map)
    
    # 添加新闻节点（无特征）
    data['news'].x = torch.ones(len(news_map), 1, dtype=torch.float)  # 占位特征
    data['news'].num_nodes = len(news_map)
    
    # 添加S-S边
    ss_edge_src = []
    ss_edge_dst = []
    ss_edge_attr = []
    for _, row in ss_edge_df.iterrows():
        if row['stock_1'] in stock_map and row['stock_2'] in stock_map:
            src = stock_map[row['stock_1']]
            dst = stock_map[row['stock_2']]
            ss_edge_src.append(src)
            ss_edge_dst.append(dst)
            ss_edge_attr.append(row['linkage_value'])
    
    if ss_edge_src:
        data['stock', 'link', 'stock'].edge_index = torch.tensor([ss_edge_src, ss_edge_dst], dtype=torch.long)
        data['stock', 'link', 'stock'].edge_attr = torch.tensor(ss_edge_attr, dtype=torch.float)
    
    # 添加N-S边和I-S边
    ns_edge_src = []
    ns_edge_dst = []
    ns_edge_attr = []
    is_edge_src = []
    is_edge_dst = []
    is_edge_attr = []
    
    for _, row in ns_is_edge_df.iterrows():
        stock_code = row['stock_code']
        industry_code = row['industry_code']
        news_id = row['id']
        
        if stock_code in stock_map and industry_code in industry_map and news_id in news_map:
            stock_idx = stock_map[stock_code]
            industry_idx = industry_map[industry_code]
            news_idx = news_map[news_id]
            
            # N-S边
            ns_edge_src.append(news_idx)
            ns_edge_dst.append(stock_idx)
            ns_edge_attr.append(row['sentiment_confidence'])
            
            # I-S边
            is_edge_src.append(industry_idx)
            is_edge_dst.append(stock_idx)
            is_edge_attr.append(row['market_ratio'])
    
    if ns_edge_src:
        data['news', 'sentiment', 'stock'].edge_index = torch.tensor([ns_edge_src, ns_edge_dst], dtype=torch.long)
        data['news', 'sentiment', 'stock'].edge_attr = torch.tensor(ns_edge_attr, dtype=torch.float)
    
    if is_edge_src:
        data['industry', 'marketratio', 'stock'].edge_index = torch.tensor([is_edge_src, is_edge_dst], dtype=torch.long)
        data['industry', 'marketratio', 'stock'].edge_attr = torch.tensor(is_edge_attr, dtype=torch.float)
    
    print(f"日期 {date}: 股票节点={len(stocks)} 新闻节点={len(news_ids)} 行业节点={len(industries)}")
    print(f"    SS边={len(ss_edge_src)} NS边={len(ns_edge_src)} IS边={len(is_edge_src)}")
    
    return data

# 3. 构建每日图数据集
train_graphs = []
test_graphs = []

for date in all_dates:
    graph = create_daily_graph(date)
    if graph is not None:
        if date in train_dates:
            train_graphs.append(graph)
        else:
            test_graphs.append(graph)

print(f"\n构建完成: {len(train_graphs)}个训练图, {len(test_graphs)}个测试图")



# 4. 创建批次数据加载器
batch_size = 16  # 可以根据GPU内存调整

train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)
# 修改模型部分
class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers):
        super().__init__()
        
        # 初始化线性变换
        self.lin_stock = Linear(8, hidden_channels)  # 股票特征固定为8维
        self.lin_news = Linear(1, hidden_channels)   # 新闻特征固定为1维
        self.lin_industry = Linear(1, hidden_channels)  # 行业特征固定为1维
        
        # 初始化HeteroConv层
        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            # 创建边类型到卷积层的映射
            conv_dict = {
                ('stock', 'link', 'stock'): SAGEConv((-1, -1), hidden_channels),
                ('news', 'sentiment', 'stock'): SAGEConv((-1, -1), hidden_channels),
                ('industry', 'marketratio', 'stock'): SAGEConv((-1, -1), hidden_channels),
            }
            conv = HeteroConv(conv_dict, aggr='mean')
            self.convs.append(conv)
        
        # 最终分类器
        self.classifier = Linear(hidden_channels, out_channels)
        
    def forward(self, x_dict, edge_index_dict):
        # 特征变换
        x_dict = {
            'stock': F.leaky_relu(self.lin_stock(x_dict['stock'])),
            'news': F.leaky_relu(self.lin_news(x_dict['news'])),
            'industry': F.leaky_relu(self.lin_industry(x_dict['industry']))
        }
        
        # 应用卷积层
        for conv in self.convs:
            # 获取当前卷积层支持的所有边类型
            conv_edge_types = list(conv.convs.keys())
            
            # 创建有效的边索引字典（只包含当前卷积层支持的边类型）
            valid_edge_index_dict = {}
            for edge_type in conv_edge_types:
                if edge_type in edge_index_dict:
                    valid_edge_index_dict[edge_type] = edge_index_dict[edge_type]
            
            # 只处理存在的边类型
            if valid_edge_index_dict:
                x_dict = conv(x_dict, valid_edge_index_dict)
                x_dict = {key: F.leaky_relu(x) for key, x in x_dict.items()}
        
        # 只返回股票节点预测
        return self.classifier(x_dict['stock'])
    
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_samples = 0
    
    for batch_idx, batch in enumerate(loader):
        try:
            # 确保所有节点类型都有特征
            for node_type in ['stock', 'news', 'industry']:
                if not hasattr(batch[node_type], 'x') or batch[node_type].x is None:
                    # 添加占位特征
                    num_nodes = batch[node_type].num_nodes
                    batch[node_type].x = torch.ones(num_nodes, 1, dtype=torch.float)
            
            # 创建边索引字典
            edge_index_dict = {}
            for edge_type in [
                ('stock', 'link', 'stock'),
                ('news', 'sentiment', 'stock'),
                ('industry', 'marketratio', 'stock')
            ]:
                if hasattr(batch[edge_type], 'edge_index') and batch[edge_type].edge_index is not None:
                    edge_index_dict[edge_type] = batch[edge_type].edge_index
            
            # 移动到设备
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # 前向传播
            out = model(batch.x_dict, edge_index_dict)
            
            # 计算损失
            y = batch['stock'].y
            mask = (y >= 0) & (y <= 1)  # 只处理有效标签 (0 或 1)
            
            if mask.sum() > 0:
                loss = criterion(out[mask], y[mask].long())
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item() * mask.sum().item()
                total_samples += mask.sum().item()
                print(f"批次 {batch_idx+1} 完成, 损失: {loss.item():.4f}")
            else:
                print(f"批次 {batch_idx+1} 跳过: 无有效标签")
                
        except Exception as e:
            print(f"处理批次 {batch_idx+1} 时出错: {str(e)}")
            # 打印详细错误信息
            print(f"节点特征: {batch.x_dict}")
            print(f"边索引字典: {edge_index_dict}")
            print(f"标签: {y}")
            if 'out' in locals():
                print(f"输出形状: {out.shape}")
            raise
    
    return total_loss / total_samples if total_samples > 0 else 0


# 7. 评估函数
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            
            y = batch['stock'].y
            mask = y >= 0  # 只评估有标签的节点
            
            if mask.sum() > 0:
                # 计算损失
                loss = criterion(out[mask], y[mask])
                total_loss += loss.item() * mask.sum().item()
                total_samples += mask.sum().item()
                
                # 收集预测和标签
                preds = out.argmax(dim=1)[mask].cpu().numpy()
                probs = F.softmax(out, dim=1)[mask, 1].cpu().numpy()
                labels = y[mask].cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels)
    
    # 计算指标
    acc = accuracy_score(all_labels, all_preds) if all_labels else 0
    f1 = f1_score(all_labels, all_preds) if all_labels else 0
    auc = roc_auc_score(all_labels, all_preds) if len(np.unique(all_labels)) > 1 else 0.5
    
    avg_loss = total_loss / total_samples if total_samples > 0 else 0
    return avg_loss, acc, f1, auc

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 初始化模型
    model = HeteroGNN(
        hidden_channels=32,
        out_channels=2,
        num_layers=2
    ).to(device)
    
    # 打印模型结构
    print("模型结构:")
    print(model)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()
    
    # 在单个批次上测试前向传播
    print("\n在单个批次上测试前向传播...")
    test_batch = next(iter(train_loader))
    
    # 准备边索引字典
    edge_index_dict = {}
    for edge_type in [
        ('stock', 'link', 'stock'),
        ('news', 'sentiment', 'stock'),
        ('industry', 'marketratio', 'stock')
    ]:
        if hasattr(test_batch[edge_type], 'edge_index') and test_batch[edge_type].edge_index is not None:
            edge_index_dict[edge_type] = test_batch[edge_type].edge_index.to(device)  # 将边索引移动到设备上
    
    # 确保特征存在
    for node_type in ['stock', 'news', 'industry']:
        if not hasattr(test_batch[node_type], 'x') or test_batch[node_type].x is None:
            num_nodes = test_batch[node_type].num_nodes
            test_batch[node_type].x = torch.ones(num_nodes, 1, dtype=torch.float).to(device)  # 将占位特征移动到设备上
    
    test_batch = test_batch.to(device)
    
    try:
        out = model(test_batch.x_dict, edge_index_dict)
        print("前向传播成功! 输出形状:", out.shape)
    except Exception as e:
        print("前向传播失败:", str(e))
        # 打印详细张量信息
        print("节点特征:")
        for k, v in test_batch.x_dict.items():
            print(f"  {k}: {v.shape}")
        
        print("边索引字典:")
        for k, v in edge_index_dict.items():
            print(f"  {k}: {v.shape}")
        
        return
    
    # 完整训练
    print("\n开始训练...")
    for epoch in range(1, 101):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc, test_f1, test_auc = evaluate(model, test_loader, device)
        
        if epoch % 1 == 0:
            print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, '
                  f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')

if __name__ == "__main__":
    main()